# Load DQ Rules — Product table

Registers the `product` table in `dq.dq_source_table` and loads 5 DQ rules into `dq.dq_rule`.

* Attach this notebook to the Lakehouse that holds the `dq` schema (run `DDL/00`–`08` first).
* Idempotent: re-running updates existing rows (MERGE) instead of duplicating them.
* Edit the **Config** cell to match your workspace, lakehouse and product column names.

In [ ]:
# ---- Config ----------------------------------------------------------------
WORKSPACE_NAME       = "MyWorkspace"
LAKEHOUSE_NAME       = "MyLakehouse"
SCHEMA_NAME          = "dbo"
TABLE_NAME           = "product"
LAYER                = "silver"          # bronze | silver | gold
BUSINESS_KEY_COLUMNS = "product_id"
PASS_THRESHOLD       = 80.00
EFFECTIVE_FROM       = "2026-11-14"
RULE_VERSION         = 1

## 1. Register the Product table

In [ ]:
from pyspark.sql import functions as F

existing = spark.sql(f"""
    SELECT table_id FROM dq.dq_source_table
    WHERE workspace_name = '{WORKSPACE_NAME}' AND lakehouse_name = '{LAKEHOUSE_NAME}'
      AND schema_name = '{SCHEMA_NAME}' AND table_name = '{TABLE_NAME}'
""").collect()

if existing:
    TABLE_ID = existing[0].table_id
else:
    TABLE_ID = spark.sql("SELECT COALESCE(MAX(table_id), 0) + 1 AS id FROM dq.dq_source_table").first().id

spark.sql(f"""
MERGE INTO dq.dq_source_table AS t
USING (SELECT {TABLE_ID} AS table_id) AS s
ON t.table_id = s.table_id
WHEN MATCHED THEN UPDATE SET
    t.layer = '{LAYER}',
    t.business_key_columns = '{BUSINESS_KEY_COLUMNS}',
    t.pass_threshold = CAST({PASS_THRESHOLD} AS DECIMAL(5,2)),
    t.is_active = true
WHEN NOT MATCHED THEN INSERT
    (table_id, workspace_name, lakehouse_name, schema_name, table_name, layer,
     business_key_columns, pass_threshold, is_active, created_ts)
VALUES
    ({TABLE_ID}, '{WORKSPACE_NAME}', '{LAKEHOUSE_NAME}', '{SCHEMA_NAME}', '{TABLE_NAME}', '{LAYER}',
     '{BUSINESS_KEY_COLUMNS}', CAST({PASS_THRESHOLD} AS DECIMAL(5,2)), true, current_timestamp())
""")
print(f"Product table registered with table_id = {TABLE_ID}")

## 2. Define the 5 rules

`rule_expression` is a Spark SQL predicate evaluated per record — `TRUE` means pass. Weights sum to 1.0.

| # | Rule | Dimension | Severity | Critical |
|---|------|-----------|----------|----------|
| 1 | Product ID is present | Completeness | Critical | Yes |
| 2 | Product name is present | Completeness | High | No |
| 3 | List price is non-negative | Validity | High | No |
| 4 | Category is in the allowed list | Validity | Low | No |
| 5 | Sell end date is not before sell start date | Consistency | Low | No |

In [ ]:
# dimension_id: 1 Completeness, 2 Validity, 3 Uniqueness, 4 Timeliness, 5 Consistency, 6 Accuracy
rules = [
    # (rule_name, dimension_id, column_name, rule_expression, weight, severity, is_critical)
    ("Product ID is present", 1, "product_id",
     "product_id IS NOT NULL", 0.30, "Critical", True),
    ("Product name is present", 1, "product_name",
     "product_name IS NOT NULL AND trim(product_name) <> ''", 0.20, "High", False),
    ("List price is non-negative", 2, "list_price",
     "list_price IS NOT NULL AND list_price >= 0", 0.25, "High", False),
    ("Category is in the allowed list", 2, "category",
     "category IN ('Bikes', 'Components', 'Clothing', 'Accessories')", 0.15, "Low", False),
    ("Sell end date is not before sell start date", 5, None,
     "sell_end_date IS NULL OR sell_end_date >= sell_start_date", 0.10, "Low", False),
]

assert abs(sum(r[4] for r in rules) - 1.0) < 1e-9, "Rule weights must sum to 1.0"

## 3. Validate expressions against the Product table (optional)

In [ ]:
src = f"{SCHEMA_NAME}.{TABLE_NAME}"
if spark.catalog.tableExists(src):
    df = spark.table(src)
    for name, _, _, expr, *_ in rules:
        df.select(F.expr(expr)).limit(1).collect()   # raises if the expression is invalid
        print(f"OK  {name}")
else:
    print(f"{src} not found in the attached lakehouse - skipping validation")

## 4. Load rules into `dq.dq_rule`

In [ ]:
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
                               DecimalType, BooleanType)
from decimal import Decimal

schema = StructType([
    StructField("rule_name", StringType(), False),
    StructField("dimension_id", IntegerType(), False),
    StructField("column_name", StringType(), True),
    StructField("rule_expression", StringType(), False),
    StructField("weight", DecimalType(9, 4), False),
    StructField("severity", StringType(), False),
    StructField("is_critical", BooleanType(), False),
])
rules_df = spark.createDataFrame(
    [(n, d, c, e, Decimal(str(w)), s, crit) for n, d, c, e, w, s, crit in rules], schema)

# Keep existing rule_ids; assign new ids after the current max
max_id = spark.sql("SELECT COALESCE(MAX(rule_id), 0) AS m FROM dq.dq_rule").first().m
existing_ids = spark.sql(f"""
    SELECT rule_name, rule_id FROM dq.dq_rule
    WHERE table_id = {TABLE_ID} AND rule_version = {RULE_VERSION}
""")

from pyspark.sql.window import Window
staged = (rules_df.join(existing_ids, "rule_name", "left")
    .withColumn("_new", F.row_number().over(Window.orderBy("rule_name")) + F.lit(max_id))
    .withColumn("rule_id", F.coalesce("rule_id", "_new").cast("int"))
    .drop("_new")
    .withColumn("table_id", F.lit(TABLE_ID).cast("int"))
    .withColumn("rule_version", F.lit(RULE_VERSION).cast("int"))
    .withColumn("effective_from", F.to_date(F.lit(EFFECTIVE_FROM)))
    .withColumn("effective_to", F.lit(None).cast("date"))
    .withColumn("is_active", F.lit(True)))
staged.createOrReplaceTempView("product_rules_stage")

spark.sql("""
MERGE INTO dq.dq_rule AS t
USING product_rules_stage AS s
ON t.rule_id = s.rule_id
WHEN MATCHED THEN UPDATE SET
    t.dimension_id = s.dimension_id, t.column_name = s.column_name,
    t.rule_expression = s.rule_expression, t.weight = s.weight,
    t.severity = s.severity, t.is_critical = s.is_critical,
    t.effective_from = s.effective_from, t.effective_to = s.effective_to,
    t.is_active = s.is_active
WHEN NOT MATCHED THEN INSERT
    (rule_id, table_id, dimension_id, rule_name, column_name, rule_expression, weight,
     severity, is_critical, rule_version, effective_from, effective_to, is_active)
VALUES
    (s.rule_id, s.table_id, s.dimension_id, s.rule_name, s.column_name, s.rule_expression, s.weight,
     s.severity, s.is_critical, s.rule_version, s.effective_from, s.effective_to, s.is_active)
""")

## 5. Verify

In [ ]:
display(spark.sql(f"""
    SELECT r.rule_id, r.rule_name, d.dimension_name, r.column_name, r.rule_expression,
           r.weight, r.severity, r.is_critical, r.rule_version, r.is_active
    FROM dq.dq_rule r
    JOIN dq.dq_dimension d ON d.dimension_id = r.dimension_id
    WHERE r.table_id = {TABLE_ID}
    ORDER BY r.rule_id
"""))